# **Milestone 5**

## **Introduction**

Master the competition metric, optimise inference, and combine models for maximum performance. Introduce TTA.

Suggested Readings:
* PyTorch - Softmax Function
* Hugging Face - Transformer
* Ensemble Learning

Competition link: https://www.kaggle.com/competitions/smart-mcq-solver-challenge

## **Initialization**

Setup (Run Before Attempting Questions) :

Use the following two fine-tuned sequence classification checkpoints:

DeBERTa: microsoft/deberta-v3-small (fine-tuned checkpoint)

RoBERTa: roberta-base (fine-tuned checkpoint)

Label Mapping
The models output logits for five labels corresponding to the answer options:

Label ID         Option

0                       A

1                       B

2                       C

3                       D

4                       E

In [1]:
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForMultipleChoice, set_seed

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train = pd.read_csv("/content/train.csv")
test = pd.read_csv("/content/test.csv")

tok_deb = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small")
mod_deb = AutoModelForMultipleChoice.from_pretrained("microsoft/deberta-v3-small").to(device)
mod_deb.eval()

tok_rob = AutoTokenizer.from_pretrained("roberta-base")
mod_rob = AutoModelForMultipleChoice.from_pretrained("roberta-base").to(device)
mod_rob.eval()

label_letters = ["A", "B", "C", "D", "E"]

def get_logits(row, tokenizer, model):
    prompt = str(row["prompt"])
    options = [str(row[l]) for l in label_letters]
    prompts = [prompt] * 5

    tokens = tokenizer(prompts, options, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
    input_ids = tokens["input_ids"].unsqueeze(0).to(device)
    attention_mask = tokens["attention_mask"].unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    return outputs.logits[0]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias  

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForMultipleChoice LOAD REPORT from: roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | MISSING    | 
roberta.pooler.dense.weight | MISSING    | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## **Question 01**

Load the fine-tuned DeBERTa and RoBERTa models.

For the prompt at row index 25, perform inference using each model independently and apply Softmax to obtain class probabilities.

Which answer option receives the highest probability from the DeBERTa model, and what is that probability?

(answer format : eg - A, probability of A)

In [2]:
row_25 = train.iloc[25]
logits_deb_25 = get_logits(row_25, tok_deb, mod_deb)
probs_deb_25 = F.softmax(logits_deb_25, dim=-1)

max_prob_deb, max_idx_deb = torch.max(probs_deb_25, dim=0)
print(f"{label_letters[max_idx_deb.item()]}, {round(max_prob_deb.item(), 4)}")

E, 0.2003


## **Question 02**

Using the same sample (row index 25), average the class probabilities from both models.

Average Probability = [P(DeBERTa) + P(RoBERTa)]/2

Which answer option receives the highest averaged probability after simple probability ensembling?

In [3]:
logits_rob_25 = get_logits(row_25, tok_rob, mod_rob)
probs_rob_25 = F.softmax(logits_rob_25, dim=-1)

avg_probs_25 = (probs_deb_25 + probs_rob_25) / 2.0
max_prob_avg, max_idx_avg = torch.max(avg_probs_25, dim=0)
print(label_letters[max_idx_avg.item()])

E


## **Question 03**

Apply weighted probability averaging after Softmax using the following weights:

DeBERTa: 0.70

RoBERTa: 0.30

Compute:

P(final) = [0.7 × P(DeBERTa)] + [0.3 × P(RoBERTa)]

Which answer option is ranked first after weighted ensembling?

In [4]:
weighted_probs_25 = (0.7 * probs_deb_25) + (0.3 * probs_rob_25)
max_prob_weighted, max_idx_weighted = torch.max(weighted_probs_25, dim=0)
print(label_letters[max_idx_weighted.item()])

E


## **Question 04**

Using the weighted ensemble probabilities from Q3, rank all five answer options.

Write the final prediction exactly in Kaggle submission format.

What is the Top-3 prediction string for row index 25?

Example : C A E

In [5]:
top3_idx = torch.topk(weighted_probs_25, 3).indices
top3_letters = [label_letters[i] for i in top3_idx]
print(" ".join(top3_letters))

E D A


## **Question 05**

Run the weighted ensemble pipeline on every row of test.csv.

Save the predictions in a file named submission.csv using the required Kaggle format:

id,prediction

where the prediction column contains the Top-3 ranked options separated by spaces.

Exactly how many prediction rows are present in the generated file (excluding the header)?

In [6]:
def get_logits_batch(batch_df, tokenizer, model):
    prompts = []
    options = []
    for _, row in batch_df.iterrows():
        p = str(row["prompt"])
        for l in label_letters:
            prompts.append(p)
            options.append(str(row[l]))

    tokens = tokenizer(prompts, options, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
    input_ids = tokens["input_ids"].to(device).view(len(batch_df), 5, 128)
    attention_mask = tokens["attention_mask"].to(device).view(len(batch_df), 5, 128)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    return outputs.logits

sub_ids = []
sub_preds = []
batch_size = 16

for i in range(0, len(test), batch_size):
    batch = test.iloc[i:i+batch_size]

    l_deb = get_logits_batch(batch, tok_deb, mod_deb)
    l_rob = get_logits_batch(batch, tok_rob, mod_rob)

    p_deb = F.softmax(l_deb, dim=-1)
    p_rob = F.softmax(l_rob, dim=-1)

    p_w = (0.7 * p_deb) + (0.3 * p_rob)
    top3_idx = torch.topk(p_w, 3, dim=-1).indices

    for j in range(len(batch)):
        top3_letters = [label_letters[idx] for idx in top3_idx[j]]
        idx_val = batch.iloc[j]["id"] if "id" in batch.columns else (i + j)
        sub_ids.append(idx_val)
        sub_preds.append(" ".join(top3_letters))

submission = pd.DataFrame({"id": sub_ids, "prediction": sub_preds})
submission.to_csv("submission.csv", index=False)
print(len(submission))

500


## **Question 06**

For the first 50 rows of test.csv, create two versions of every prompt:

1.Original prompt

2.Instruction-augmented prompt by prepending: "Answer the following multiple-choice question carefully:"

Run inference using DeBERTa on both versions.

Average the predicted probabilities from both passes.

How many of the first 50 rows produce a different Top-1 prediction after applying Test-Time Augmentation?

In [7]:
changed_tta = 0
batch_size = 16

for i in range(0, 50, batch_size):
    batch = test.iloc[i:min(i+batch_size, 50)]

    l_orig = get_logits_batch(batch, tok_deb, mod_deb)
    p_orig = F.softmax(l_orig, dim=-1)
    top1_orig = torch.argmax(p_orig, dim=-1)

    batch_tta = batch.copy()
    batch_tta["prompt"] = "Answer the following multiple-choice question carefully: " + batch_tta["prompt"].astype(str)

    l_tta = get_logits_batch(batch_tta, tok_deb, mod_deb)
    p_tta = F.softmax(l_tta, dim=-1)

    p_avg = (p_orig + p_tta) / 2.0
    top1_tta = torch.argmax(p_avg, dim=-1)

    changed_tta += (top1_orig != top1_tta).sum().item()

print(changed_tta)

9


## **Question 07**

Process the first 100 rows of test.csv. And compare the Top-1 prediction from:

1. DeBERTa

2. Weighted Ensemble

How many rows have different Top-1 predictions?

In [8]:
changed_top1 = 0
batch_size = 16

for i in range(0, 100, batch_size):
    batch = test.iloc[i:min(i+batch_size, 100)]

    l_deb = get_logits_batch(batch, tok_deb, mod_deb)
    l_rob = get_logits_batch(batch, tok_rob, mod_rob)

    p_deb = F.softmax(l_deb, dim=-1)
    p_rob = F.softmax(l_rob, dim=-1)
    p_w = (0.7 * p_deb) + (0.3 * p_rob)

    top1_deb = torch.argmax(p_deb, dim=-1)
    top1_w = torch.argmax(p_w, dim=-1)

    changed_top1 += (top1_deb != top1_w).sum().item()

print(changed_top1)

51


## **Question 08**

For the first 100 rows of test.csv, record the highest class probability (confidence) predicted by:

1. DeBERTa

2. Weighted Ensemble

For every row, compute:

Confidence Gain = Ensemble Confidence−DeBERTa Confidence

How many rows have a positive confidence gain (greater than 0)?

In [9]:
positive_gain = 0
batch_size = 16

for i in range(0, 100, batch_size):
    batch = test.iloc[i:min(i+batch_size, 100)]

    l_deb = get_logits_batch(batch, tok_deb, mod_deb)
    l_rob = get_logits_batch(batch, tok_rob, mod_rob)

    p_deb = F.softmax(l_deb, dim=-1)
    p_rob = F.softmax(l_rob, dim=-1)
    p_w = (0.7 * p_deb) + (0.3 * p_rob)

    conf_deb, _ = torch.max(p_deb, dim=-1)
    conf_w, _ = torch.max(p_w, dim=-1)

    positive_gain += (conf_w > conf_deb).sum().item()

print(positive_gain)

61


## **Question 09**

For the first 100 rows of test.csv, compare the Top-3 prediction strings generated by:

1. DeBERTa alone

2. Weighted Ensemble

How many rows have at least one change in their ordered Top-3 ranking after ensembling?

Examples:

A C D vs. A D C

In [10]:
changed_top3 = 0
batch_size = 16

for i in range(0, 100, batch_size):
    batch = test.iloc[i:min(i+batch_size, 100)]

    l_deb = get_logits_batch(batch, tok_deb, mod_deb)
    l_rob = get_logits_batch(batch, tok_rob, mod_rob)

    p_deb = F.softmax(l_deb, dim=-1)
    p_rob = F.softmax(l_rob, dim=-1)
    p_w = (0.7 * p_deb) + (0.3 * p_rob)

    top3_deb = torch.topk(p_deb, 3, dim=-1).indices
    top3_w = torch.topk(p_w, 3, dim=-1).indices

    for j in range(len(batch)):
        if top3_deb[j].tolist() != top3_w[j].tolist():
            changed_top3 += 1

print(changed_top3)

92


## **Question 10**

Using the Top-3 predictions generated by your weighted ensemble for the first 100 validation samples, compute the MAP@3 score.

What is the final MAP@3 score?

(Round to 4 decimal places.)

In [11]:
def map_at_3(truth, preds):
    if truth == preds[0]:
        return 1.0
    elif truth == preds[1]:
        return 1.0/2.0
    elif truth == preds[2]:
        return 1.0/3.0
    return 0.0

map_scores = []
batch_size = 16

for i in range(0, 100, batch_size):
    batch = train.iloc[i:min(i+batch_size, 100)]

    l_deb = get_logits_batch(batch, tok_deb, mod_deb)
    l_rob = get_logits_batch(batch, tok_rob, mod_rob)

    p_deb = F.softmax(l_deb, dim=-1)
    p_rob = F.softmax(l_rob, dim=-1)
    p_w = (0.7 * p_deb) + (0.3 * p_rob)

    top3_idx = torch.topk(p_w, 3, dim=-1).indices

    for j in range(len(batch)):
        truth = batch.iloc[j]["answer"]
        top3_letters = [label_letters[idx] for idx in top3_idx[j]]
        map_scores.append(map_at_3(truth, top3_letters))

print(round(sum(map_scores) / len(map_scores), 4))

0.4067
